# Policy heads: basic information

Reads the checkpoints used in the paper and reports, for each policy, the total size and the part we call the **head** (the only part retrained in Step 1):

| Policy | Head | Where |
|---|---|---|
| ACT | `model.action_head` (Linear 512 → 10) | lerobot `model.safetensors` |
| Diffusion Policy | every `cond_encoder` (FiLM: Mish → Linear) in the U-Net | diffusion_policy `.ckpt`, EMA weights |
| SmolVLA | `model.action_out_proj` (Linear 720 → 32) | lerobot `model.safetensors` |

Kernel: **Python (evh-vla-venv)**. Run from anywhere in the repo; nothing is written.

In [1]:
import json, glob, os, gc
from pathlib import Path
import dill, torch, pandas as pd
from safetensors import safe_open

ROOT = Path.cwd()
while not (ROOT / 'paper').exists():
    ROOT = ROOT.parent
I = ROOT / 'outputs/interp'

ACT = {'Lift': I / 'aug/train/act_lift_aug_base', 'Square': I / 'square/train/act_square_abs6',
       'ToolHang': I / 'toolhang/train/act_toolhang_abs6'}
ACT = {t: p / 'checkpoints/last/pretrained_model' for t, p in ACT.items()}
DP = {'Lift': ROOT / 'checkpoints/dp_lift_ph_image_cnn.ckpt', 'Square': ROOT / 'checkpoints/dp_square_ph_image_cnn.ckpt',
      'ToolHang': I / 'toolhang/dp/run/checkpoints/latest.ckpt'}
VLA = {'Lift': I / 'remote1/smolvla_final', 'Square': I / 'remote5/smolvla_square_final'}

def shapes(path):
    """{key: shape} of a safetensors file, without loading the tensors."""
    with safe_open(str(path), 'pt') as f:
        return {k: tuple(f.get_slice(k).get_shape()) for k in f.keys()}

def numel(shape):
    n = 1
    for d in shape:
        n *= d
    return n

rows = []

## ACT

In [2]:
for task, p in ACT.items():
    s = shapes(p / 'model.safetensors')
    cfg = json.load(open(p / 'config.json'))
    head = {k: v for k, v in s.items() if 'action_head' in k}
    rows.append(dict(policy='ACT', task=task, total=sum(map(numel, s.values())),
                     head=sum(map(numel, head.values())), head_keys=head,
                     chunk=cfg['chunk_size'], action_dim=cfg['output_features']['action']['shape'][0]))
    print(task, head, '| dim_model', cfg['dim_model'], '| chunk', cfg['chunk_size'])

Lift {'model.action_head.bias': (10,), 'model.action_head.weight': (10, 512)} | dim_model 512 | chunk 16
Square {'model.action_head.bias': (10,), 'model.action_head.weight': (10, 512)} | dim_model 512 | chunk 16
ToolHang {'model.action_head.bias': (10,), 'model.action_head.weight': (10, 512)} | dim_model 512 | chunk 16


### 5,130 or 5,220?

The head is `10 × 512` weights + `10` biases = **5,130**. Every ACT head retrain in the paper refits these (closed form on rollouts), and their logs report 5,130.

A few runs instead used the older gradient path in `scripts/interp_edit.py`, which trains a rank-*r* update `U V` on the weight only. At rank 10 that has `10 × 10 + 10 × 512` = **5,220** trainable numbers, but it still only changes the same 5,120-entry weight matrix. The ACT position control in Table 2 (`x_head_r10_b1_unsh`) is one of those runs.

In [3]:
print('full head:', 10 * 512 + 10, '| rank-10 factors:', 10 * 10 + 10 * 512)
logs = []
for f in sorted(glob.glob(str(I / 'edit/*.log'))):
    lines = [l for l in open(f) if l.startswith('{')]
    if not lines:
        continue
    st = json.loads(lines[-1])
    if 'act_' in str(st.get('parent', '')) and st.get('params') in (5130, 5220):
        logs.append(dict(run=Path(f).stem, params=st['params'], labels=st.get('labels'),
                         onpolicy=bool(st.get('onpolicy')), closed_form=st.get('closed_form')))
pd.DataFrame(logs).sort_values('params', ascending=False).head(8)

full head: 5130 | rank-10 factors: 5220


,run,params,labels,onpolicy,closed_form
12,sq_depth_head_r4,5220,shifted,False,False
29,x_head_r10_b1_unsh,5220,unshifted,False,False
11,sq_depth_head,5220,shifted,False,False
0,rep_bothcam,5130,shifted,True,False
3,rep_darkredwroll,5130,shifted,True,False
2,rep_bothzoom,5130,shifted,True,False
1,rep_bothflip,5130,shifted,True,False
6,rep_invisible,5130,shifted,True,False


## Diffusion Policy

Loads each `.ckpt` (EMA weights). The ToolHang one also holds the optimizer state, so it is 4.6 GB and takes longer; the normalizer is excluded from the totals.

In [4]:
for task, p in DP.items():
    if not p.exists():
        print(task, 'missing:', p)
        continue
    payload = torch.load(open(p, 'rb'), pickle_module=dill, map_location='cpu', weights_only=False)
    cfg, sd = payload['cfg'], payload['state_dicts']['ema_model']
    film = {k: tuple(v.shape) for k, v in sd.items() if 'cond_encoder' in k}
    rows.append(dict(policy='Diffusion', task=task,
                     total=sum(v.numel() for k, v in sd.items() if not k.startswith('normalizer')),
                     head=sum(numel(v) for v in film.values()), head_keys=f'{len(film)} tensors in cond_encoder',
                     chunk=cfg.horizon, action_dim=cfg.task.shape_meta.action.shape[0]))
    print(task, '| horizon', cfg.horizon, 'n_obs', cfg.n_obs_steps, 'n_action', cfg.n_action_steps,
          '| obs', list(cfg.task.shape_meta.obs.keys()))
    print('   first FiLM layers:', dict(list(film.items())[:2]))
    del payload, sd
    gc.collect()

Lift | horizon 16 n_obs 2 n_action 8 | obs ['agentview_image', 'robot0_eye_in_hand_image', 'robot0_eef_pos', 'robot0_eef_quat', 'robot0_gripper_qpos']
   first FiLM layers: {'model.mid_modules.0.cond_encoder.1.weight': (4096, 402), 'model.mid_modules.0.cond_encoder.1.bias': (4096,)}


Square | horizon 16 n_obs 2 n_action 8 | obs ['agentview_image', 'robot0_eye_in_hand_image', 'robot0_eef_pos', 'robot0_eef_quat', 'robot0_gripper_qpos']
   first FiLM layers: {'model.mid_modules.0.cond_encoder.1.weight': (4096, 402), 'model.mid_modules.0.cond_encoder.1.bias': (4096,)}


ToolHang | horizon 16 n_obs 2 n_action 8 | obs ['robot0_eef_pos', 'robot0_eef_quat', 'robot0_eye_in_hand_image', 'robot0_gripper_qpos', 'agentview_image']
   first FiLM layers: {'model.mid_modules.0.cond_encoder.1.weight': (4096, 402), 'model.mid_modules.0.cond_encoder.1.bias': (4096,)}


## SmolVLA

In [5]:
for task, p in VLA.items():
    s = shapes(p / 'model.safetensors')
    cfg = json.load(open(p / 'config.json'))
    head = {k: v for k, v in s.items() if 'action_out_proj' in k}
    rows.append(dict(policy='SmolVLA', task=task, total=sum(map(numel, s.values())),
                     head=sum(map(numel, head.values())), head_keys=head,
                     chunk=cfg['chunk_size'], action_dim=cfg['output_features']['action']['shape'][0]))
    print(task, head, '| padded action dim', cfg['max_action_dim'], '| VLM', cfg['vlm_model_name'])
    print('   instruction:', (p / 'instruction.txt').read_text().strip() if (p / 'instruction.txt').exists() else '-')

Lift {'model.action_out_proj.bias': (32,), 'model.action_out_proj.weight': (32, 720)} | padded action dim 32 | VLM HuggingFaceTB/SmolVLM2-500M-Video-Instruct
   instruction: lift the cube
Square {'model.action_out_proj.bias': (32,), 'model.action_out_proj.weight': (32, 720)} | padded action dim 32 | VLM HuggingFaceTB/SmolVLM2-500M-Video-Instruct
   instruction: put the square nut on the square peg


## Summary

In [6]:
df = pd.DataFrame(rows)
df['head %'] = (100 * df['head'] / df['total']).round(3)
df[['policy', 'task', 'total', 'head', 'head %', 'chunk', 'action_dim']]

,policy,task,total,head,head %,chunk,action_dim
0,ACT,Lift,51589840,5130,0.010,16,10
1,ACT,Square,51589840,5130,0.010,16,10
2,ACT,ToolHang,51589840,5130,0.010,16,10
3,Diffusion,Lift,300392150,11554816,3.847,16,10
4,Diffusion,Square,300392150,11554816,3.847,16,10
5,Diffusion,ToolHang,300392150,11554816,3.847,16,10
6,SmolVLA,Lift,450046234,23072,0.005,16,10
7,SmolVLA,Square,450046234,23072,0.005,16,10
